# Spatial Regularization for Low-Statistics Neutron Imaging

Demonstrates the progression of fitting methods for neutron resonance
imaging at decreasing photon count levels:

1. **LM (Levenberg-Marquardt)** — fast and accurate at high counts,
   fails catastrophically below I₀ ≈ 50
2. **Poisson KL** — correct noise model, works at I₀ = 2–10,
   but noisy for ill-conditioned multi-isotope cases
3. **Poisson KL + Spatial Regularization** — uses Fisher eigenbasis
   to selectively smooth poorly-determined parameter directions,
   recovering density maps that would otherwise be too noisy

The key test case is **Hafnium** (6 isotopes), where Hf-180 (35%
abundant) has essentially no resonance features — the solver has
no spectral handle for it. Spatial regularization stabilizes Hf-180
while leaving well-determined isotopes untouched.

## Prerequisites

```bash
pixi run build
```

**Previous:** [Spatial Mapping Demo](01_spatial_mapping_demo.ipynb)

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import nereids

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 150

## 1. Setup: Hafnium 6-Isotope Phantom

Natural hafnium has 6 stable isotopes with very different resonance
structures:

| Isotope | Abundance | σ_max (barns) | Resonance character |
|---------|-----------|----------------|--------------------|
| Hf-174 | 0.16% | 14,421 | Strong, well-resolved |
| Hf-176 | 5.26% | 26,794 | Strong |
| Hf-177 | 18.60% | 52,638 | Dense, many resonances |
| Hf-178 | 27.28% | 116,584 | Single giant resonance |
| Hf-179 | 13.62% | 7,926 | Moderate |
| **Hf-180** | **35.08%** | **16** | **Featureless** — nearly flat |

In [ ]:
# Load all 6 Hf isotopes
hf_isotopes = [
    (72, 174, 'Hf-174', 0.0016),
    (72, 176, 'Hf-176', 0.0526),
    (72, 177, 'Hf-177', 0.1860),
    (72, 178, 'Hf-178', 0.2728),
    (72, 179, 'Hf-179', 0.1362),
    (72, 180, 'Hf-180', 0.3508),
]

isotopes = [nereids.load_endf(z, a) for z, a, _, _ in hf_isotopes]
names = [n for _, _, n, _ in hf_isotopes]
total_density = 0.01  # atoms/barn total
true_densities = [abd * total_density for _, _, _, abd in hf_isotopes]

print('Hafnium natural composition:')
for name, td in zip(names, true_densities):
    print(f'  {name}: {td:.6f} atoms/barn')

# Energy grid: 1-200 eV, 1500 bins (realistic VENUS)
energies = np.linspace(1.0, 200.0, 1500)
print(f'\nEnergy grid: {len(energies)} bins, {energies[0]:.0f}-{energies[-1]:.0f} eV')

In [ ]:
# Generate synthetic transmission and add Poisson noise
t_1d = np.asarray(nereids.forward_model(
    energies, list(zip(isotopes, true_densities)), temperature_k=293.6))

H, W = 16, 16
rng = np.random.default_rng(42)

def make_noisy_data(t_1d, height, width, i0, rng):
    """Generate a noisy 3D transmission cube from a 1D spectrum."""
    t_clean = np.tile(t_1d[:, None, None], (1, height, width))
    counts = rng.poisson(i0 * t_clean).astype(np.float64)
    trans = counts / i0
    unc = np.sqrt(np.maximum(counts, 1.0)) / i0
    return trans, unc, counts

# Generate data at multiple count levels
datasets = {}
for i0 in [100, 10, 5]:
    trans, unc, counts = make_noisy_data(t_1d, H, W, i0, rng)
    datasets[i0] = (trans, unc, counts)
    n_zero = (counts == 0).mean() * 100
    print(f'I\u2080={i0:>3}: mean counts/bin={counts.mean():.1f}, '
          f'zero-count bins={n_zero:.0f}%')

## 2. LM Fitting: Good at High Counts, Fails at Low

Levenberg-Marquardt assumes Gaussian noise (χ² minimization). This is
a valid approximation when counts per bin are above ~50 (Central Limit
Theorem). Below that, the Poisson distribution is asymmetric and
zero-count bins become common — LM cannot handle these correctly.

In [ ]:
print('LM fitting across count levels:')
print(f'{"I\u2080":>5} {"Isotope":<10} {"True":>10} {"Mean":>10} {"Bias%":>8} {"Std":>10}')
print('-' * 58)

lm_results = {}
for i0 in [100, 10, 5]:
    trans, unc, _ = datasets[i0]
    result = nereids.spatial_map(
        trans, unc, energies, isotopes,
        temperature_k=293.6, max_iter=200)
    lm_results[i0] = result
    
    for k, (name, true_n) in enumerate(zip(names, true_densities)):
        d = np.asarray(result.density_maps[k])
        bias = (d.mean() - true_n) / true_n * 100
        print(f'{i0:>5} {name:<10} {true_n:>10.6f} {d.mean():>10.6f} '
              f'{bias:>+8.1f} {d.std():>10.6f}')
    print()

## 3. Poisson KL: Correct Noise Model, Works at Low Counts

The Poisson-KL fitter minimizes the Kullback-Leibler divergence between
observed and expected counts. It correctly handles zero-count bins and
maintains low bias even at I₀ = 5.

However, for **ill-conditioned** cases like Hf-180 (featureless isotope),
the per-pixel variance remains high because the solver has no spectral
handle to constrain the density.

In [ ]:
print('Poisson KL fitting across count levels:')
print(f'{"I\u2080":>5} {"Isotope":<10} {"True":>10} {"Mean":>10} {"Bias%":>8} {"Std":>10} {"Std/True%":>10}')
print('-' * 68)

kl_results = {}
for i0 in [100, 10, 5]:
    trans, unc, counts = datasets[i0]
    ob = np.full_like(counts, float(i0))
    result = nereids.spatial_map(
        counts, ob, energies, isotopes,
        temperature_k=293.6, max_iter=200, fitter='poisson')
    kl_results[i0] = result
    
    for k, (name, true_n) in enumerate(zip(names, true_densities)):
        d = np.asarray(result.density_maps[k])
        bias = (d.mean() - true_n) / true_n * 100
        std_pct = d.std() / true_n * 100
        print(f'{i0:>5} {name:<10} {true_n:>10.6f} {d.mean():>10.6f} '
              f'{bias:>+8.1f} {d.std():>10.6f} {std_pct:>10.0f}%')
    print()

## 4. Spatial Regularization: Recovering Ill-Conditioned Isotopes

`spatial_map_regularized` adds a physics-informed spatial smoothing step
after the per-pixel KL fitting. It uses the **Fisher information matrix**
(computed from the known cross-sections) to identify which parameter
directions are well-determined and which are poorly-determined.

- **Well-determined directions** (strong resonances, high Fisher info):
  left untouched — the per-pixel data is sufficient.
- **Poorly-determined directions** (featureless isotopes, weak Fisher info):
  spatially smoothed across neighboring pixels.

This selectivity comes from the **known physics** (ENDF cross-sections),
not from the noisy data.

In [ ]:
print('Poisson KL + Spatial Regularization:')
print(f'{"I\u2080":>5} {"Isotope":<10} {"KL Std":>10} {"Reg Std":>10} {"Reduction":>10} {"Reg Bias%":>10}')
print('-' * 58)

reg_results = {}
for i0 in [100, 10, 5]:
    trans, unc, counts = datasets[i0]
    
    t0 = time.perf_counter()
    # spatial_map_regularized with fitter='poisson' uses Poisson NLL
    # on normalized transmission (not raw counts). Pass trans/unc,
    # same as the LM path — the solver handles the noise model.
    result = nereids.spatial_map_regularized(
        trans, unc, energies, isotopes,
        temperature_k=293.6, max_iter=200,
        fitter='poisson',
        threshold=0.05, smooth_iter=10,
        compute_uncertainty=True)
    dt = time.perf_counter() - t0
    reg_results[i0] = result
    
    kl = kl_results[i0]
    for k, (name, true_n) in enumerate(zip(names, true_densities)):
        d_kl = np.asarray(kl.density_maps[k])
        d_reg = np.asarray(result.density_maps[k])
        ratio = d_kl.std() / d_reg.std() if d_reg.std() > 0 else float('inf')
        bias = (d_reg.mean() - true_n) / true_n * 100
        print(f'{i0:>5} {name:<10} {d_kl.std():>10.6f} {d_reg.std():>10.6f} '
              f'{ratio:>9.1f}x {bias:>+10.1f}')
    
    print(f'      n_weak_directions={result.n_weak_directions}, '
          f'eigenvalues={[f"{e:.0f}" for e in result.fisher_eigenvalues]}, '
          f'time={dt:.2f}s')
    print()

## 5. Uncertainty Estimation

The regularized result includes per-pixel uncertainty maps from the
Laplace approximation (inverse Hessian of the penalized objective).
These account for both the data noise and the spatial regularization.

In [ ]:
# Check uncertainty coverage at I0=5
result = reg_results[5]
print('Uncertainty coverage at I\u2080=5 (expect ~95%):')
for k, (name, true_n) in enumerate(zip(names, true_densities)):
    d = np.asarray(result.density_maps[k])
    u = np.asarray(result.uncertainty_maps[k])
    in_ci = np.abs(d - true_n) < 1.96 * u
    coverage = in_ci.mean() * 100
    print(f'  {name}: mean_unc={u.mean():.2e}, 95% CI coverage={coverage:.0f}%')

## 6. Visual Comparison

Side-by-side density maps for Hf-180 (the hardest case) at I₀=5.

In [ ]:
# Hf-180 density maps: LM vs KL vs KL+Regularization at I0=5
idx_180 = 5  # Hf-180 is the 6th isotope
true_n = true_densities[idx_180]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vmin, vmax = 0, true_n * 3

for ax, (label, result) in zip(axes, [
    ('LM (I\u2080=5)', lm_results[5]),
    ('Poisson KL (I\u2080=5)', kl_results[5]),
    ('KL + Regularization (I\u2080=5)', reg_results[5]),
]):
    d = np.asarray(result.density_maps[idx_180])
    bias = (d.mean() - true_n) / true_n * 100
    im = ax.imshow(d, cmap='viridis', vmin=vmin, vmax=vmax,
                   interpolation='nearest')
    ax.set_title(f'{label}\nmean={d.mean():.4f}, std={d.std():.4f}\n'
                 f'bias={bias:+.1f}%', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, label='atoms/barn')

fig.suptitle(f'Hf-180 (featureless, true={true_n:.4f} atoms/barn)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# All 6 isotopes: KL vs Regularized at I0=5
fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for k, (name, true_n) in enumerate(zip(names, true_densities)):
    d_kl = np.asarray(kl_results[5].density_maps[k])
    d_reg = np.asarray(reg_results[5].density_maps[k])
    vmax = true_n * 2.5
    
    im0 = axes[0, k].imshow(d_kl, cmap='viridis', vmin=0, vmax=vmax,
                             interpolation='nearest')
    axes[0, k].set_title(f'{name}\nKL std={d_kl.std():.2e}', fontsize=9)
    axes[0, k].set_xticks([]); axes[0, k].set_yticks([])
    
    im1 = axes[1, k].imshow(d_reg, cmap='viridis', vmin=0, vmax=vmax,
                             interpolation='nearest')
    ratio = d_kl.std() / d_reg.std() if d_reg.std() > 0 else float('inf')
    axes[1, k].set_title(f'Reg std={d_reg.std():.2e}\n{ratio:.1f}\u00d7 reduction',
                          fontsize=9)
    axes[1, k].set_xticks([]); axes[1, k].set_yticks([])

axes[0, 0].set_ylabel('Poisson KL', fontsize=11)
axes[1, 0].set_ylabel('KL + Regularization', fontsize=11)
fig.suptitle('All 6 Hf isotopes at I\u2080=5: KL vs Regularized', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Summary

| Method | When to use | Hf-180 performance (I\u2080=5) |
|--------|-------------|------------------------------|
| LM | I\u2080 > 50 (Gaussian regime) | Catastrophically biased |
| Poisson KL | I\u2080 = 2\u201350 (Poisson regime) | Unbiased but noisy |
| KL + Regularization | I\u2080 = 2\u201350, multi-isotope | Unbiased AND smooth |

### Key properties of the regularization:

- **Selective**: only smooths poorly-determined directions (Hf-180),
  leaves well-determined ones (Hf-174) untouched
- **Physics-driven**: the Fisher eigenbasis threshold comes from ENDF
  cross-sections, not from the noisy data
- **Uncertainty-aware**: provides per-pixel error bars via Laplace
  approximation
- **Single parameter**: threshold=0.05 works across isotopes and count levels